In [1]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

In [2]:
import gradio as gr
import sqlite3
import re
from datetime import datetime
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

In [3]:
schema = {
    "users": ["id","name","email","signup_date","age","country","status"],
    "orders": ["id","user_id","product_name","amount","order_date","status"],
    "products": ["id","name","price","category","stock"],
    "transactions": ["id","user_id","amount","date","type"]
}

In [4]:
dangerous_keywords = ["DROP","DELETE","ALTER","TRUNCATE","INSERT","UPDATE"]

def is_safe(query):
    for word in dangerous_keywords:
        if word.lower() in query.lower():
            return False
    return True

In [5]:
def get_table_name(text):
    text=text.lower()

    if "user" in text:
        return "users"
    elif "order" in text or "purchase" in text:
        return "orders"
    elif "product" in text:
        return "products"
    elif "transaction" in text or "payment" in text:
        return "transactions"

    return "users"

In [6]:
import re

def generate_sql(prompt):

    text = prompt.lower()
    table = get_table_name(text)

    condition = ""

    # greater than
    match = re.search(r'greater than (\d+)', text)
    if match:
        value = match.group(1)
        condition = f"amount > {value}"

    # less than
    match = re.search(r'less than (\d+)', text)
    if match:
        value = match.group(1)
        condition = f"amount < {value}"

    # last week
    if "last week" in text:
        if condition:
            condition += " AND DATE(date) >= DATE('now','-7 days')"
        else:
            condition = "DATE(date) >= DATE('now','-7 days')"

    # last month
    if "last month" in text:
        if table == "users":
            condition = "DATE(signup_date) >= DATE('now','-1 month')"
        else:
            condition = "DATE(date) >= DATE('now','-1 month')"

    # country filter
    match = re.search(r'from (\w+)', text)
    if match:
        country = match.group(1)
        condition = f"country = '{country}'"

    if condition:
        sql = f"SELECT * FROM {table} WHERE {condition} LIMIT 10"
    else:
        sql = f"SELECT * FROM {table} LIMIT 10"

    return sql

In [7]:
def chat(prompt):
    return generate_sql(prompt)

demo = gr.Interface(
    fn=chat,
    inputs="text",
    outputs="text",
    title="AI SQL Query Generator"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://21f7dce462bde4c2e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
